# Simulación Cuántica en GPU: cuStateVec y Qiskit Aer

**Laboratorio 30 · Nivel muy avanzado**

Exploramos cómo escalar simulaciones cuánticas usando GPUs con NVIDIA cuStateVec
y el backend `AerSimulator` de Qiskit con aceleración GPU.

## 1. ¿Por qué GPU para simulación cuántica?

La simulación de un circuito de n qubits requiere manipular un vector de estado
de 2^n amplitudes complejas (cada una: 16 bytes en float128):

| n qubits | Memoria (float64) | Tiempo CPU (puerta) | Límite práctico |
|----------|-------------------|---------------------|------------------|
| 20       | 16 MB             | ~1 ms               | Laptop           |
| 25       | 512 MB            | ~30 ms              | Servidor CPU     |
| 30       | 16 GB             | ~1 s                | GPU (A100 80GB)  |
| 34       | 256 GB            | ~16 s               | Multi-GPU        |
| 40+      | >4 TB             | minutos-horas       | HPC cluster      |

Una GPU A100 tiene:
- 80 GB de VRAM (vs ~8-32 GB GPU normal)
- 312 TFLOPS FP16 / 19.5 TFLOPS FP64
- Bandwidth de memoria: 2 TB/s (vs ~50 GB/s en CPU)

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import QFT
from qiskit.quantum_info import Statevector

# Verificar backends disponibles
try:
    import custatevec
    HAS_GPU = True
    print('✅ cuStateVec disponible — GPU habilitada')
except ImportError:
    HAS_GPU = False
    print('⚠️  cuStateVec no disponible — modo CPU (instalar qiskit-aer-gpu para GPU)')

print(f'Qiskit Aer version: {AerSimulator().version}')

## 2. Configuración del simulador con backend GPU

Qiskit Aer detecta automáticamente si hay GPU disponible con `qiskit-aer-gpu`.

In [ ]:
def get_simulator(gpu: bool = False, method: str = 'statevector') -> AerSimulator:
    """
    Devuelve un simulador Aer configurado para CPU o GPU.
    
    method options:
      'statevector'       — simulación exacta, exponencial en memoria
      'matrix_product_state' — MPS, eficiente para estados de baja entanglement
      'extended_stabilizer'  — para circuitos casi-Clifford
    """
    if gpu and HAS_GPU:
        return AerSimulator(method=method, device='GPU', cuStateVec_enable=True)
    else:
        return AerSimulator(method=method, device='CPU')

sim_cpu = get_simulator(gpu=False)
sim_gpu = get_simulator(gpu=HAS_GPU)

print(f'Simulador CPU: {sim_cpu.name()}, método: {sim_cpu.options.method}')
print(f'Simulador GPU: {sim_gpu.name()}, método: {sim_gpu.options.method}')

## 3. Benchmark: QFT en CPU vs GPU

In [ ]:
def benchmark_qft(n_qubits: int, backend: AerSimulator, shots: int = 0) -> float:
    """
    Mide el tiempo de simulación de QFT con n_qubits.
    shots=0: simulación de statevector exacta (sin medir).
    """
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))  # superposición inicial
    qft = QFT(n_qubits, do_swaps=True)
    qc.append(qft, range(n_qubits))
    if shots > 0:
        qc.measure_all()

    qc_t = transpile(qc, backend, optimization_level=0)

    t0 = time.perf_counter()
    job = backend.run(qc_t, shots=max(1, shots))
    _ = job.result()
    return time.perf_counter() - t0

# Benchmark para diferentes tamaños
n_range = list(range(4, 22, 2))
tiempos_cpu = []
tiempos_gpu = []

print(f'Benchmark QFT: CPU vs GPU')
print(f'{"n":>4} | {"CPU (s)":>10} | {"GPU (s)":>10} | {"Speedup":>10}')
print('-' * 42)

for n in n_range:
    t_cpu = benchmark_qft(n, sim_cpu)
    tiempos_cpu.append(t_cpu)

    if HAS_GPU:
        t_gpu = benchmark_qft(n, sim_gpu)
        tiempos_gpu.append(t_gpu)
        speedup = t_cpu / t_gpu
        print(f'{n:>4} | {t_cpu:>10.4f} | {t_gpu:>10.4f} | {speedup:>10.2f}×')
    else:
        tiempos_gpu.append(None)
        print(f'{n:>4} | {t_cpu:>10.4f} |   (sin GPU) |        N/A')

In [ ]:
# Gráfica de tiempos
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].semilogy(n_range, tiempos_cpu, 'b-o', lw=2, ms=6, label='CPU')
if HAS_GPU and any(t is not None for t in tiempos_gpu):
    axes[0].semilogy(n_range, tiempos_gpu, 'r-s', lw=2, ms=6, label='GPU')
    # Referencia teórica: GPU ~100× más rápido para >20 qubits
    t_ref = [t * 0.02 for t in tiempos_cpu]
    axes[0].semilogy(n_range, t_ref, 'r--', alpha=0.4, label='GPU esperado (×50)')

axes[0].set_xlabel('Número de qubits')
axes[0].set_ylabel('Tiempo (s)')
axes[0].set_title('Tiempo de simulación QFT')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Memoria requerida vs n qubits
n_mem = np.arange(10, 36)
mem_gb = 2**n_mem * 16 / 1e9  # 16 bytes por amplitud compleja float128

axes[1].semilogy(n_mem, mem_gb, 'g-', lw=2)
axes[1].axhline(8,   color='b', ls='--', alpha=0.7, label='8 GB (GPU consumidor)')
axes[1].axhline(80,  color='r', ls='--', alpha=0.7, label='80 GB (A100 SXM)')
axes[1].axhline(512, color='purple', ls='--', alpha=0.7, label='512 GB (multi-GPU)')
axes[1].set_xlabel('Número de qubits')
axes[1].set_ylabel('Memoria requerida (GB)')
axes[1].set_title('Memoria para simulación statevector exacta')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Matrix Product States (MPS): alternativa eficiente

Para circuitos con entrelazamiento limitado, MPS escala polinomialmente en
lugar de exponencialmente. El parámetro clave es `bond_dimension` χ:

- Memoria: O(n·χ²) en lugar de O(2^n)
- Exacto para χ = 2^(n/2) (límite de entrelazamiento máximo)
- Eficiente cuando χ << 2^(n/2)

In [ ]:
def benchmark_mps_vs_sv(n_qubits: int, n_layers: int = 3) -> dict:
    """
    Compara MPS vs statevector para un circuito de n qubits.
    Usa un circuito de capas aleatorias para controlar el entrelazamiento.
    """
    rng = np.random.default_rng(42)

    # Circuito con entrelazamiento moderado (capas nearest-neighbor)
    qc = QuantumCircuit(n_qubits)
    for layer in range(n_layers):
        # Capa de rotaciones aleatorias
        for q in range(n_qubits):
            qc.ry(rng.uniform(0, np.pi), q)
        # Capa de CNOTs nearest-neighbor
        for q in range(0, n_qubits - 1, 2):
            qc.cx(q, q+1)
        for q in range(1, n_qubits - 1, 2):
            qc.cx(q, q+1)

    # Simular con statevector
    t0 = time.perf_counter()
    sim_sv = AerSimulator(method='statevector')
    qc_sv = transpile(qc, sim_sv)
    sv_job = sim_sv.run(qc_sv, shots=1).result()
    t_sv = time.perf_counter() - t0

    # Simular con MPS
    t0 = time.perf_counter()
    sim_mps = AerSimulator(method='matrix_product_state',
                           matrix_product_state_truncation_threshold=1e-8)
    qc_mps = transpile(qc, sim_mps)
    mps_job = sim_mps.run(qc_mps, shots=1).result()
    t_mps = time.perf_counter() - t0

    return {
        'n': n_qubits,
        't_sv': t_sv,
        't_mps': t_mps,
        'speedup_mps': t_sv / t_mps,
    }

print('MPS vs Statevector (circuito nearest-neighbor, 3 capas):')
print(f'{"n":>4} | {"SV (s)":>8} | {"MPS (s)":>8} | {"Speedup MPS":>12}')
print('-' * 40)
for n in [8, 12, 16, 20]:
    r = benchmark_mps_vs_sv(n, n_layers=3)
    print(f'{n:>4} | {r["t_sv"]:>8.3f} | {r["t_mps"]:>8.3f} | {r["speedup_mps"]:>12.2f}×')

## 5. Simulación multi-GPU y distribución del vector de estado

Para n > 30 qubits, un solo GPU no tiene suficiente memoria. La solución es
distribuir el vector de estado entre múltiples GPUs.

In [ ]:
import numpy as np

def analisis_multi_gpu(n_max: int = 40) -> None:
    """Análisis de capacidad multi-GPU para simulación cuántica."""
    configs = [
        ('1× A100 (80 GB)',   80,    1),
        ('4× A100 (320 GB)',  320,   4),
        ('8× A100 (640 GB)',  640,   8),
        ('16× H100 (1.28 TB)', 1280, 16),
    ]

    print('Capacidad de simulación por configuración GPU:')
    print(f'{"Configuración":>25} | {"VRAM (GB)":>10} | {"Max qubits (FP64)":>18} | {"Max qubits (FP32)"}')
    print('-' * 80)

    n_vals_by_config = {}
    for nombre, vram_gb, n_gpus in configs:
        # FP64: 16 bytes por amplitud
        n_fp64 = int(np.floor(np.log2(vram_gb * 1e9 / 16)))
        # FP32: 8 bytes por amplitud
        n_fp32 = int(np.floor(np.log2(vram_gb * 1e9 / 8)))
        print(f'{nombre:>25} | {vram_gb:>10} | {n_fp64:>18} | {n_fp32}')
        n_vals_by_config[nombre] = (n_fp64, n_fp32, vram_gb)

    # Comparar con clásico supercomputador
    print(f'{"Fugaku (460 TB RAM)":>25} | {460000:>10} | {int(np.log2(460000e9/16)):>18} | -')

analisis_multi_gpu()

# Visualización
fig, ax = plt.subplots(figsize=(10, 5))

n_qubits = np.arange(10, 45)
mem_fp64 = 2**n_qubits * 16 / 1e9  # GB
mem_fp32 = 2**n_qubits * 8 / 1e9   # GB

ax.fill_between(n_qubits, mem_fp32, mem_fp64, alpha=0.2, color='blue', label='FP32-FP64 range')
ax.semilogy(n_qubits, mem_fp64, 'b-', lw=2, label='FP64 (doble precisión)')
ax.semilogy(n_qubits, mem_fp32, 'b--', lw=2, label='FP32 (simple precisión)')

limits = [
    (80,    'r',  '1× A100 (80 GB)'),
    (320,   'g',  '4× A100 (320 GB)'),
    (640,   'm',  '8× A100 (640 GB)'),
    (1280,  'orange', '16× H100 (1.28 TB)'),
]
for mem, color, label in limits:
    ax.axhline(mem, color=color, ls='--', alpha=0.7, lw=1.5, label=label)

ax.set_xlabel('Número de qubits')
ax.set_ylabel('Memoria requerida (GB)')
ax.set_title('Memoria para simulación statevector vs configuración GPU')
ax.legend(fontsize=8, loc='upper left')
ax.grid(alpha=0.3)
ax.set_xlim(10, 44)
plt.tight_layout()
plt.show()

## 6. cuStateVec: API de bajo nivel

`cuStateVec` es la librería NVIDIA de bajo nivel para manipulación de vectores
de estado cuántico en GPU. Qiskit Aer la usa internamente.

In [ ]:
# Demostración del modelo de uso de cuStateVec (pseudo-código documentado)
# La API real requiere CUDA y cuStateVec instalados

custatevec_pseudocode = """
# cuStateVec — API de bajo nivel (NVIDIA CUDA)
# Requiere: pip install qiskit-aer-gpu (incluye cuStateVec)

import cupy as cp
import custatevec as cusv

n_qubits = 25
dim = 2**n_qubits

# Inicializar vector de estado en GPU (|0...0⟩)
sv = cp.zeros(dim, dtype=cp.complex64)
sv[0] = 1.0

# Crear handle de cuStateVec
handle = cusv.create()

# Aplicar puerta H en qubit 0
h_matrix = cp.array([[1, 1], [1, -1]], dtype=cp.complex64) / np.sqrt(2)
cusv.apply_matrix(
    handle,
    sv.data.ptr,            # puntero al vector en GPU
    cusv.CUDA_C_32F,        # tipo de dato
    n_qubits,               # número de qubits
    h_matrix.data.ptr,      # puntero a la matriz de la puerta
    cusv.CUDA_C_32F,
    cusv.MATRIX_LAYOUT_ROW,
    0,                      # qubit objetivo
    [],                     # qubits de control
    [],                     # valores de control
    cusv.COMPUTE_32F,
)

# Muestrear el vector de estado
n_shots = 1000
bitstrings = cusv.sampler_run(handle, sv.data.ptr, ..., n_shots)

cusv.destroy(handle)
"""

print('API cuStateVec de bajo nivel:')
print(custatevec_pseudocode)

print('\nEn la práctica: Qiskit Aer abstrae toda esta complejidad.')
print('Solo necesitas: AerSimulator(method="statevector", device="GPU", cuStateVec_enable=True)')

## 7. Circuitos grandes con AerSimulator GPU

In [ ]:
def circuito_random_layers(n_qubits: int, n_layers: int, seed: int = 0) -> QuantumCircuit:
    """Circuito aleatorio de n capas con puertas de 1 y 2 qubits."""
    from qiskit.circuit.random import random_circuit
    return random_circuit(n_qubits, n_layers, max_operands=2, seed=seed)

# Simular circuito de 20 qubits con 10 capas
n = 20
qc_grande = circuito_random_layers(n, n_layers=10, seed=42)
qc_grande.measure_all()

print(f'Circuito aleatorio: {n} qubits, {qc_grande.depth()} profundidad, {qc_grande.size()} puertas')

# Simular con CPU
t0 = time.perf_counter()
sim_cpu_run = AerSimulator(method='statevector')
qc_t = transpile(qc_grande, sim_cpu_run)
result_cpu = sim_cpu_run.run(qc_t, shots=1024).result()
t_cpu = time.perf_counter() - t0

counts_cpu = result_cpu.get_counts()
print(f'\nCPU: {t_cpu:.3f} s, {len(counts_cpu)} resultados distintos')

if HAS_GPU:
    t0 = time.perf_counter()
    sim_gpu_run = AerSimulator(method='statevector', device='GPU', cuStateVec_enable=True)
    qc_t_gpu = transpile(qc_grande, sim_gpu_run)
    result_gpu = sim_gpu_run.run(qc_t_gpu, shots=1024).result()
    t_gpu = time.perf_counter() - t0
    print(f'GPU: {t_gpu:.3f} s  (speedup: ×{t_cpu/t_gpu:.1f})')

## 8. Herramientas alternativas de simulación GPU

| Herramienta | Backend | Max qubits | Ventaja | Limitación |
|-------------|---------|-----------|---------|------------|
| Qiskit Aer GPU | NVIDIA cuStateVec | ~30 (1 GPU) | Integrado con Qiskit | Solo NVIDIA |
| cuQuantum Appliance | NVIDIA | 34+ (multi-GPU) | Rendimiento máximo | Configuración compleja |
| PennyLane + Lightning.gpu | NVIDIA | ~30 | Diferenciable | Solo NVIDIA |
| Amazon Braket DM1 | AWS | 34 | Sin hardware local | Pago por uso |
| Google Cirq + qsim | GPU/TPU | 40 | Muy eficiente | Integración manual |
| Stim | CPU (Clifford) | 10^6+ | Mega-escala en Clifford | Solo Clifford |
| QuEST | Multi-GPU/HPC | 40+ | Open source | C API |
| Intel Quantum Simulator | CPU AVX | 32 | Muy optimizado | Solo CPU |

In [ ]:
# Comparativa de métodos disponibles en Aer
from qiskit_aer import AerSimulator

metodos_aer = [
    ('statevector',           'Exacto, exponencial en memoria',       30,  True),
    ('density_matrix',        'Simulación ruidosa exacta',            15,  True),
    ('matrix_product_state',  'MPS, eficiente con bajo entrelazamiento', 100, False),
    ('stabilizer',            'Clifford exacto, polinomial',          1e6, False),
    ('extended_stabilizer',   'Casi-Clifford, Monte Carlo',           50,  False),
    ('unitary',               'Calcula la unitaria completa',         24,  True),
    ('superop',               'Superoperador completo',               12,  True),
]

print('Métodos de simulación disponibles en Qiskit Aer:')
print(f'{"Método":>30} | {"Descripción":>42} | {"Max q aprox":>12} | {"GPU?"}')
print('-' * 100)
for metodo, desc, max_q, gpu in metodos_aer:
    gpu_str = '✅' if gpu else '—'
    max_q_str = f'{int(max_q):,}' if max_q < 1e5 else f'{max_q:.0e}'
    print(f'{metodo:>30} | {desc:>42} | {max_q_str:>12} | {gpu_str}')

## 9. Perfil de rendimiento: dónde va el tiempo

In [ ]:
import cProfile
import pstats
import io

def profile_simulation(n_qubits: int = 15) -> None:
    """Perfila una simulación de statevector para identificar cuellos de botella."""
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))
    for i in range(n_qubits - 1):
        qc.cx(i, i+1)
    qc.measure_all()

    sim = AerSimulator(method='statevector')
    qc_t = transpile(qc, sim)

    profiler = cProfile.Profile()
    profiler.enable()
    _ = sim.run(qc_t, shots=100).result()
    profiler.disable()

    # Mostrar top 10 funciones por tiempo
    stream = io.StringIO()
    stats = pstats.Stats(profiler, stream=stream)
    stats.sort_stats('cumulative')
    stats.print_stats(10)
    print(stream.getvalue())

profile_simulation(15)

## 10. Resumen: elegir el método de simulación correcto

```
¿Cuántos qubits?  ──► <25: statevector CPU es suficiente
                   ──► 25-32: statevector GPU (A100)
                   ──► >32: MPS (si bajo entanglement) o multi-GPU

¿Qué tipo de circuito?
  Clifford puro ─────► stabilizer (hasta 10^6 qubits)
  Bajo entanglement ──► MPS (hasta 100+ qubits)
  General ────────────► statevector (exacto, limitado por memoria)
  Ruidoso ────────────► density_matrix (hasta ~15 qubits exacto)

¿Tienes GPU NVIDIA?
  Sí: AerSimulator(device='GPU', cuStateVec_enable=True)
  No: AerSimulator(device='CPU') + paralelización con threads
```